In [13]:
# load the cleaned bangalore_df from cleaned_data/bangalore_cleaned.csv
import numpy as np
import pandas as pd
df = pd.read_csv("cleaned_data/bangalore_cleaned.csv") # removed the missing value records

In [14]:
bangalore_df = df.copy()

In [15]:
bangalore_df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170,2.0,1.0,38.00
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785,5.0,3.0,295.00


In [16]:
# sample 10 rows of the dataframe
bangalore_df.sample(10)

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
4153,Super built-up Area,Ready To Move,Whitefield,2 BHK,Sonviik,1186,2.0,3.0,55.0
718,Built-up Area,Ready To Move,Kaggadasapura,4 BHK,Adsia F,2150,4.0,1.0,100.0
5584,Super built-up Area,20-Dec,Jakkur,2 BHK,ArndsSk,1282,2.0,1.0,72.0
3591,Super built-up Area,Ready To Move,Kadugodi,1 BHK,RatatHa,840 - 1010,1.0,0.0,40.7
6855,Super built-up Area,Ready To Move,Sarjapur Road,3 BHK,PraleSi,2140,3.0,2.0,139.0
5853,Super built-up Area,Ready To Move,Sarjapur,3 BHK,Sondaka,1333,3.0,1.0,50.0
4095,Super built-up Area,18-Dec,Old Airport Road,4 BHK,Jaades,2690,4.0,3.0,201.0
298,Super built-up Area,Ready To Move,Surabhi Layout,3 BHK,VDdonel,1753,3.0,2.0,120.0
2576,Built-up Area,Ready To Move,Sarjapur Road,2 BHK,Shthi S,915,2.0,2.0,37.0
2341,Super built-up Area,Ready To Move,Gopalkrishna Nagar,3 BHK,MJarlPe,1530,2.0,1.0,97.0


total_sqft has some values that cannot be directly converted to numeric. They are like "1133 - 1384". We can take the average of these two numbers as an estimate of the total_sqft for these entries.

In [17]:
bangalore_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


In [18]:
# total_sqft is in object format and there are some non-numeric values in the column.The values are in the format of "number - number". We will process them.

def convert_sqft(x):
    try:
        if "-" in str(x):
            a,b = x.split("-")
            return (float(a) + float(b))/2 # take the average of the two numbers
        return float(x)
    except:
        return None


bangalore_df["total_sqft"] = bangalore_df["total_sqft"].apply(convert_sqft)


In [19]:
bangalore_df.isnull().sum()

area_type        0
availability     0
location         0
size             0
society          0
total_sqft      15
bath             0
balcony          0
price            0
dtype: int64

The missing values on inspection were due to some non numeric values that were is various formats like 1333sqft, 1333sq.ft, 1333sq ft, 1333sqft. Since they are very few, we can remove these entries overall.

In [20]:
# dropping the rows with null values in total_sqft column
bangalore_df = bangalore_df.dropna(subset=["total_sqft"])

In [21]:
bangalore_df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price'],
      dtype='object')

In [22]:
bangalore_df['size'].sample(10)

242     2 BHK
4591    3 BHK
5899    2 BHK
3167    3 BHK
2717    3 BHK
4524    3 BHK
1864    2 BHK
1816    3 BHK
5412    2 BHK
147     2 BHK
Name: size, dtype: object

There are two types of string formats present in the size column.

1. 2 BHK
2. 2 Bedroom


In [23]:
import re 

def extract_bhk(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower().strip()
    # matching patterns like "2 bhk", "2 bedroom" 
    match = re.search(r'(\d+)\s*(bhk|bedroom)' , x)

    if match:
        return int(match.group(1))
    
    return None 

bangalore_df['bhk'] = bangalore_df['size'].apply(extract_bhk)

In [24]:
bangalore_df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00,4.0
2,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00,3.0
3,Super built-up Area,Ready To Move,Whitefield,2 BHK,DuenaTa,1170.0,2.0,1.0,38.00,2.0
4,Plot Area,Ready To Move,Whitefield,4 Bedroom,Prrry M,2785.0,5.0,3.0,295.00,4.0


In [26]:
null_bhk_indices = bangalore_df.loc[bangalore_df['bhk'].isnull()].index
null_bhk_indices

Index([13, 460, 792, 1376, 1538, 2675, 2785, 2889, 4535, 5687], dtype='int64')

In [27]:
# Inspecting these rows
df.iloc[null_bhk_indices]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
13,Super built-up Area,18-Nov,Thanisandra,1 RK,Bhe 2ko,510,1.0,0.0,25.25
460,Super built-up Area,Ready To Move,Thanisandra,1 RK,Bhmesy,445,1.0,0.0,28.00
792,Super built-up Area,21-Nov,Thanisandra,1 RK,Bhe 2ko,510,1.0,1.0,25.25
1376,Super built-up Area,19-Dec,Bhoganhalli,1 RK,Rosha I,296,1.0,0.0,22.89
1538,Super built-up Area,20-Aug,Rachenahalli,1 RK,AsNowre,440,1.0,0.0,28.00
2675,Built-up Area,Ready To Move,Electronic City,1 RK,GMown E,435,1.0,1.0,19.50
2785,Built-up Area,17-Jun,Whitefield,1 RK,Prtates,905,1.0,1.0,52.00
2889,Super built-up Area,18-May,Rachenahalli,1 RK,AsNowre,385 - 440,1.0,0.0,19.80
4535,Super built-up Area,Ready To Move,Banaswadi,1 RK,Krntsee,527,1.0,0.0,35.00
5687,Built-up Area,Ready To Move,Electronic City,1 RK,GMown E,550,1.0,1.0,27.00


These indices have size in RK rather than in BHK. For simplicity, we can remove these entries as well since they are very few.

In [12]:
print(bangalore_df['bhk'].unique())

[ 2.  4.  3. nan  1.  5. 11.  9.  6.  7.]


In [13]:
# count the nan in bhk column
print(bangalore_df['bhk'].isna().sum())

10


In [28]:
# dropping the rows with null values in bhk column
bangalore_df = bangalore_df.dropna(subset=["bhk"])

In [29]:
bangalore_df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price', 'bhk'],
      dtype='object')

In [ ]:
df = bangalore_df[['location' , 'total_sqft' , 'price' ,
                   'bhk' , 'bath' , 'balcony' , 'area_type' ,
                   'availability' , 'society']]

# size feature is not required as we have extracted bhk from it and total_sqft is in numeric format now. We will drop the size column.

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7129 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   location      7129 non-null   object 
 1   total_sqft    7129 non-null   float64
 2   price         7129 non-null   float64
 3   bhk           7119 non-null   float64
 4   bath          7129 non-null   float64
 5   balcony       7129 non-null   float64
 6   area_type     7129 non-null   object 
 7   availability  7129 non-null   object 
 8   society       7129 non-null   object 
dtypes: float64(5), object(4)
memory usage: 557.0+ KB


In [17]:
df.head()

,location,total_sqft,price,bhk,bath,balcony,area_type,availability,society
0,Electronic City Phase II,1056.0,39.07,2.0,2.0,1.0,Super built-up Area,19-Dec,Coomee
1,Chikka Tirupathi,2600.0,120.00,4.0,5.0,3.0,Plot Area,Ready To Move,Theanmp
2,Lingadheeranahalli,1521.0,95.00,3.0,3.0,1.0,Super built-up Area,Ready To Move,Soiewre
3,Whitefield,1170.0,38.00,2.0,2.0,1.0,Super built-up Area,Ready To Move,DuenaTa
4,Whitefield,2785.0,295.00,4.0,5.0,3.0,Plot Area,Ready To Move,Prrry M


In [18]:
df['text'] = (
    df['bhk'].astype(str) + "BHK Property in " + df["location"] + 
    " with " + df['total_sqft'].astype(str) + "sqft, priced at " +
    df['price'].astype(str) + " lakh. " + "Area type: " + df['area_type'] + ". " + "Availability: " + df['availability'] + ". " + "Society: " + df['society'] +"Bathrooms: " + df['bath'].astype(str) + ". " + "Balcony: " + df['balcony'].astype(str) + "."
)

In [19]:
df.head()

,location,total_sqft,price,bhk,bath,balcony,area_type,availability,society,text
0,Electronic City Phase II,1056.0,39.07,2.0,2.0,1.0,Super built-up Area,19-Dec,Coomee,2.0BHK Property in Electronic City Phase II wi...
1,Chikka Tirupathi,2600.0,120.00,4.0,5.0,3.0,Plot Area,Ready To Move,Theanmp,4.0BHK Property in Chikka Tirupathi with 2600....
2,Lingadheeranahalli,1521.0,95.00,3.0,3.0,1.0,Super built-up Area,Ready To Move,Soiewre,3.0BHK Property in Lingadheeranahalli with 152...
3,Whitefield,1170.0,38.00,2.0,2.0,1.0,Super built-up Area,Ready To Move,DuenaTa,"2.0BHK Property in Whitefield with 1170.0sqft,..."
4,Whitefield,2785.0,295.00,4.0,5.0,3.0,Plot Area,Ready To Move,Prrry M,"4.0BHK Property in Whitefield with 2785.0sqft,..."
